# Gauge Gap Analysis

This notebook identifies ungauged HUC-10 watersheds by integrating stream gauge coverage using vectorized hydrography dataset (Hydrofabric). The analysis follows these steps:

1. Load spatial layers from a GeoPackage (flowpaths, divides, gages, HUC-12, HUC-10, state boundary)
2. Assign stream gages to catchment divides (point-in-polygon)
3. Integrate gage presence to flowpaths and evaluate network coverage via BFS traversal
4. Integrate coverage upward: flowpaths → divides → HUC-12 → HUC-10
5. Apply state-boundary filters to finalize ungauged HUC-10 candidates
6. Write all result layers to an output GeoPackage

Requirements: `numpy`, `pandas`, `geopandas`, `networkx`, `shapely`

## 1. Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from shapely.geometry import Point

## 2. Configuration

All file paths, layer names, column mappings, and analysis thresholds are centralized here. Edit this block before running the notebook.

In [2]:
CONFIG = {
    "input_gpkg": "data/meramec.gpkg", #path to input gpkg
    "layers": {
        "flowpaths": "flowpaths",
        "divides": "divides",
        "gages": "gages",
        "huc12": "huc12",
        "huc10": "huc10",
        "state": "state",
    },
    "flow_id_col": "id",
    "flow_toid_col": "toid",
    "drainage_area_col": "tot_draina",
    "hydroseq_col": "hydroseq",
    "divides_flow_id_col": "id",
    "divide_id_col": "id",
    "huc12_id_col": "huc12",
    "huc10_id_col": "huc10",
    "coverage_threshold_km2": 2000.0,
    "output_gpkg": "Gap_output_final_v2.gpkg",
}

## 3. Utility Functions

Shared helper functions used throughout the pipeline:
- `strip_id_prefix` — normalizes IDs like `wb-123` → `123` so joins work across layers with different ID formats
- `ensure_projected_crs` — reprojects to a metric CRS when distance/area operations are needed
- `to_yes_no` — converts a boolean Series to `"Yes"` / `"No"` string labels
- `to_numeric_safe` — coerces a Series to numeric, replacing unparseable values with `NaN`

In [3]:
def strip_id_prefix(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    s = str(value).strip()
    if "-" in s:
        s = s.split("-")[-1]
    return s


def ensure_projected_crs(gdf, fallback_epsg=3857):
    original_crs = gdf.crs
    if original_crs is None:
        return gdf.set_crs(epsg=fallback_epsg), original_crs
    if hasattr(original_crs, "is_geographic") and original_crs.is_geographic:
        return gdf.to_crs(epsg=fallback_epsg), original_crs
    return gdf, original_crs


def to_yes_no(bool_series):
    return np.where(bool_series, "Yes", "No")


def to_numeric_safe(series):
    return pd.to_numeric(series, errors="coerce")

## 4. Load Spatial Layers

All layers are read from the input GeoPackage and reprojected to a common CRS (that of the flowpaths layer).

In [4]:
def load_layers(cfg):
    gpkg_path = cfg["input_gpkg"]
    layer_names = cfg["layers"]

    flowpaths = gpd.read_file(gpkg_path, layer=layer_names["flowpaths"])
    divides   = gpd.read_file(gpkg_path, layer=layer_names["divides"])
    gages     = gpd.read_file(gpkg_path, layer=layer_names["gages"])
    huc12     = gpd.read_file(gpkg_path, layer=layer_names["huc12"])
    huc10     = gpd.read_file(gpkg_path, layer=layer_names["huc10"])
    state     = gpd.read_file(gpkg_path, layer=layer_names["state"])

    if flowpaths.crs is not None:
        target_crs = flowpaths.crs
        for gdf in (divides, gages, huc12, huc10, state):
            if gdf.crs != target_crs:
                gdf = gdf.to_crs(target_crs)

        divides = divides.to_crs(target_crs) if divides.crs != target_crs else divides
        gages   = gages.to_crs(target_crs)   if gages.crs   != target_crs else gages
        huc12   = huc12.to_crs(target_crs)   if huc12.crs   != target_crs else huc12
        huc10   = huc10.to_crs(target_crs)   if huc10.crs   != target_crs else huc10
        state   = state.to_crs(target_crs)   if state.crs   != target_crs else state

    return flowpaths, divides, gages, huc12, huc10, state


flowpaths, divides, gages, huc12, huc10, state = load_layers(CONFIG)

## 5. Assign Gages to Catchment Divides

Each gage point is assigned to the catchment divide polygon it falls within using a spatial join. 

In [5]:
def assign_points_to_polygons(points_gdf, polygons_gdf, polygon_id_col, max_nearest_dist=None):
    pts = points_gdf.copy()
    polys = polygons_gdf[[polygon_id_col, "geometry"]].copy()

    joined = gpd.sjoin(
        pts,
        polys.rename(columns={polygon_id_col: "poly_id"}),
        how="left",
        predicate="within",
    )
    joined = joined.drop(columns=[c for c in joined.columns if c.startswith("index_")], errors="ignore")

    
    return joined


gages_assigned = assign_points_to_polygons(
    points_gdf=gages,
    polygons_gdf=divides,
    polygon_id_col=CONFIG["divide_id_col"],
)

print(f"Gages assigned to divides: {gages_assigned['poly_id'].notna().sum()} of {len(gages_assigned)}")

Gages assigned to divides: 27 of 27


## 6. Flag Gaged Divides and Flowpaths

Using the gage-to-divide assignments:
- `build_divide_gage_flag` marks each divide with `has_gage = Yes/No` based on gage count
- `transfer_gage_flag_to_flowpaths` assigns this flag to the matching flowpath using stripped ID joins, creating a `gage_status` column

In [6]:
def build_divide_gage_flag(divides_gdf, gages_with_divide_gdf, divide_id_col):
    div = divides_gdf.copy()
    g2d = gages_with_divide_gdf.dropna(subset=["poly_id"]).copy()

    gage_counts = g2d.groupby("poly_id").size().rename("gage_count").reset_index()

    div = div.merge(
        gage_counts,
        left_on=divide_id_col,
        right_on="poly_id",
        how="left",
    ).drop(columns=["poly_id"], errors="ignore")

    div["gage_count"] = div["gage_count"].fillna(0).astype(int)
    div["has_gage"] = to_yes_no(div["gage_count"] > 0)
    return div


def transfer_gage_flag_to_flowpaths(flowpaths_gdf, divides_gdf, flow_id_col, divides_flow_id_col):
    fp = flowpaths_gdf.copy()
    dv = divides_gdf.copy()

    fp["id_stripped"] = fp[flow_id_col].map(strip_id_prefix)
    dv["id_stripped"] = dv[divides_flow_id_col].map(strip_id_prefix)

    gage_flag_by_id = (
        dv.groupby("id_stripped")["has_gage"]
        .apply(lambda s: "Yes" if (s == "Yes").any() else "No")
        .rename("gage_status")
        .reset_index()
    )

    fp = fp.merge(gage_flag_by_id, on="id_stripped", how="left")
    fp["gage_status"] = fp["gage_status"].fillna("No")
    return fp


divides_flagged = build_divide_gage_flag(divides, gages_assigned, CONFIG["divide_id_col"])
flowpaths_flagged = transfer_gage_flag_to_flowpaths(
    flowpaths,
    divides_flagged,
    flow_id_col=CONFIG["flow_id_col"],
    divides_flow_id_col=CONFIG["divides_flow_id_col"],
)

print(flowpaths_flagged["gage_status"].value_counts())

gage_status
No     1269
Yes      26
Name: count, dtype: int64


## 7. Compute Network Gage Coverage

A directed graph is built from the flowpath network (edges from `id` → `toid`). For each gaged flowpath node, a BFS propagation marks neighboring nodes as `well_gaged = Yes` based on drainage area bounds:

- Large gages (drainage area ≥ threshold): coverage window = `[area − threshold/2, area + threshold/2]`
- Small gages (drainage area < threshold): coverage window = `[0.5×area, 1.5×area]`

Nodes are processed in descending hydrosequence order (downstream-to-upstream). BFS stops at already-covered nodes or nodes with an existing gage.

In [7]:
def compute_network_gage_coverage(
    flowpaths_gdf,
    flow_id_col,
    flow_toid_col,
    drainage_area_col,
    hydroseq_col,
    gage_status_col="gage_status",
    coverage_threshold_km2=2000.0,
):

    fp = flowpaths_gdf.copy()

    fp["id_stripped"]   = fp[flow_id_col].map(strip_id_prefix)
    fp["toid_stripped"] = fp[flow_toid_col].map(strip_id_prefix)
    fp[drainage_area_col] = to_numeric_safe(fp[drainage_area_col])
    fp[hydroseq_col]      = to_numeric_safe(fp[hydroseq_col])
    fp["well_gaged"] = "No"

    G = nx.DiGraph()

    for idx, row in fp.iterrows():
        node_id = row["id_stripped"]
        if node_id is None:
            continue
        G.add_node(node_id, **{
            drainage_area_col: float(row[drainage_area_col]) if pd.notna(row[drainage_area_col]) else np.nan,
            gage_status_col:   row[gage_status_col],
            hydroseq_col:      float(row[hydroseq_col]) if pd.notna(row[hydroseq_col]) else np.nan,
            "df_idx":          idx,
        })

    for _, row in fp.iterrows():
        from_node = row["id_stripped"]
        to_node   = row["toid_stripped"]
        if from_node and to_node and to_node in G.nodes:
            G.add_edge(from_node, to_node)

    gaged_nodes = sorted(
        [(n, d[hydroseq_col]) for n, d in G.nodes(data=True)
         if d.get(gage_status_col) == "Yes" and pd.notna(d.get(hydroseq_col))],
        key=lambda x: x[1], reverse=True
    )

    well_gaged_set = set()

    for gaged_node, _ in gaged_nodes:
        if gaged_node in well_gaged_set:
            continue

        node_attrs = G.nodes[gaged_node]
        gage_area  = node_attrs.get(drainage_area_col, np.nan)
        if pd.isna(gage_area):
            continue

        fp.at[node_attrs["df_idx"], "well_gaged"] = "Yes"
        well_gaged_set.add(gaged_node)

        if gage_area >= coverage_threshold_km2:
            upstream_bound   = gage_area - coverage_threshold_km2 / 2.0
            downstream_bound = gage_area + coverage_threshold_km2 / 2.0
        else:
            upstream_bound   = 0.5 * gage_area
            downstream_bound = 1.5 * gage_area

        for neighbors_fn, bound, comparator in [
            (G.predecessors, upstream_bound,   lambda a, b: a >= b),
            (G.successors,   downstream_bound, lambda a, b: a <= b),
        ]:
            queue   = deque([gaged_node])
            visited = set()
            while queue:
                current = queue.popleft()
                if current in visited:
                    continue
                visited.add(current)

                if current != gaged_node:
                    if current in well_gaged_set:
                        continue
                    if G.nodes[current].get(gage_status_col) == "Yes":
                        continue
                    node_area = G.nodes[current].get(drainage_area_col, np.nan)
                    if pd.notna(node_area) and comparator(node_area, bound):
                        fp.at[G.nodes[current]["df_idx"], "well_gaged"] = "Yes"
                        well_gaged_set.add(current)
                    else:
                        continue

                for neighbor in neighbors_fn(current):
                    if neighbor not in visited:
                        queue.append(neighbor)

    return fp


flowpaths_covered = compute_network_gage_coverage(
    flowpaths_flagged,
    flow_id_col=CONFIG["flow_id_col"],
    flow_toid_col=CONFIG["flow_toid_col"],
    drainage_area_col=CONFIG["drainage_area_col"],
    hydroseq_col=CONFIG["hydroseq_col"],
    gage_status_col="gage_status",
    coverage_threshold_km2=CONFIG["coverage_threshold_km2"],
)

print(flowpaths_covered["well_gaged"].value_counts())

well_gaged
No     1181
Yes     114
Name: count, dtype: int64


## 8. Integrate Coverage: Flowpaths → Divides

A divide is marked as `covered_divide = Yes` if its associated flowpath has `well_gaged = Yes`. Matching is performed on stripped IDs. Where duplicates exist, `Yes` wins.

In [8]:
def integrate_coverage_to_divides(divides_gdf, flowpaths_gdf, divide_flow_id_col, flow_id_col):
    dv = divides_gdf.copy()
    fp = flowpaths_gdf.copy()

    dv["id_stripped"] = dv[divide_flow_id_col].map(strip_id_prefix)
    fp["id_stripped"] = fp[flow_id_col].map(strip_id_prefix)

    coverage_by_id = (
        fp.groupby("id_stripped")["well_gaged"]
        .apply(lambda s: "Yes" if (s == "Yes").any() else "No")
        .rename("covered_divide")
        .reset_index()
    )

    dv = dv.merge(coverage_by_id, on="id_stripped", how="left")
    dv["covered_divide"] = dv["covered_divide"].fillna("No")
    return dv


divides_covered = integrate_coverage_to_divides(
    divides_flagged,
    flowpaths_covered,
    divide_flow_id_col=CONFIG["divides_flow_id_col"],
    flow_id_col=CONFIG["flow_id_col"],
)

print(divides_covered["covered_divide"].value_counts())

covered_divide
No     1181
Yes     114
Name: count, dtype: int64


## 9. Integrate Coverage: Divides → HUC-12 → HUC-10

HUC-12 coverage is determined by presence of gauged divides.

HUC-10 coverage is determined by presence of gauged HUC-12.

In [9]:
def integrate_coverage_to_huc_by_area(
    divides_gdf,
    huc_gdf,
    huc_id_col,
    covered_divide_col="covered_divide",
    min_fraction=0.03,
):
    dv  = divides_gdf.copy()
    huc = huc_gdf.copy()

    covered_divides = dv[dv[covered_divide_col] == "Yes"].copy()
    if len(covered_divides) == 0:
        huc["covered_huc"]  = "No"
        huc["huc_area"]     = np.nan
        huc["covered_area"] = 0.0
        huc["covered_frac"] = 0.0
        return huc

    orig_crs = huc.crs
    huc_proj, _  = ensure_projected_crs(huc, fallback_epsg=3857)
    divs_proj, _ = ensure_projected_crs(covered_divides, fallback_epsg=3857)

    huc_proj["huc_area"] = huc_proj.geometry.area

    intersection = gpd.overlay(
        divs_proj[[covered_divide_col, "geometry"]],
        huc_proj[[huc_id_col, "geometry"]],
        how="intersection",
        keep_geom_type=False,
    )

    if len(intersection) == 0:
        huc_proj["covered_area"] = 0.0
        huc_proj["covered_frac"] = 0.0
        huc_proj["covered_huc"]  = "No"
    else:
        intersection["int_area"] = intersection.geometry.area
        covered_area = (
            intersection.groupby(huc_id_col)["int_area"]
            .sum()
            .rename("covered_area")
            .reset_index()
        )
        huc_proj = huc_proj.merge(covered_area, on=huc_id_col, how="left")
        huc_proj["covered_area"] = huc_proj["covered_area"].fillna(0.0)
        huc_proj["covered_frac"] = np.where(
            huc_proj["huc_area"] > 0,
            huc_proj["covered_area"] / huc_proj["huc_area"],
            0.0,
        )
        huc_proj["covered_huc"] = to_yes_no(huc_proj["covered_frac"] >= min_fraction)

    if orig_crs is not None and huc_proj.crs != orig_crs:
        return huc_proj.to_crs(orig_crs)
    return huc_proj


def integrate_huc12_coverage_to_huc10(huc12_covered_gdf, huc10_gdf, huc12_id_col, huc10_id_col):
    h12 = huc12_covered_gdf.copy()
    h10 = huc10_gdf.copy()

    h12_centroids = h12[[huc12_id_col, "covered_huc", "geometry"]].copy()
    h12_centroids["geometry"] = h12_centroids.geometry.centroid

    joined = gpd.sjoin(
        h12_centroids,
        h10[[huc10_id_col, "geometry"]].copy(),
        how="left",
        predicate="within",
    ).drop(columns=[c for c in h12_centroids.columns if c.startswith("index_")], errors="ignore")

    covered_huc10_ids = (
        joined.loc[joined["covered_huc"] == "Yes", huc10_id_col]
        .dropna()
        .unique()
        .tolist()
    )

    h10["covered_huc"] = to_yes_no(h10[huc10_id_col].isin(covered_huc10_ids))
    return h10


huc12_covered = integrate_coverage_to_huc_by_area(
    divides_covered, huc12, CONFIG["huc12_id_col"],
    covered_divide_col="covered_divide", min_fraction=0.02,
)

huc10_covered = integrate_huc12_coverage_to_huc10(
    huc12_covered_gdf=huc12_covered,
    huc10_gdf=huc10,
    huc12_id_col=CONFIG["huc12_id_col"],
    huc10_id_col=CONFIG["huc10_id_col"],
)

print("HUC-12 covered:", huc12_covered["covered_huc"].value_counts().to_dict())
print("HUC-10 covered:", huc10_covered["covered_huc"].value_counts().to_dict())

HUC-12 covered: {'No': 62, 'Yes': 38}
HUC-10 covered: {'Yes': 13, 'No': 5}


## 10. State Boundary

Two spatial attributes are computed for each HUC-10 relative to the state boundary:

1. **Fraction inside state** (`frac_in_state`) — the proportion of each HUC-10's area that lies within the state polygon
2. **Outlet inside state** (`outlet_in_state`) — whether the most downstream flowpath's outlet point falls within the state

These are used to filter ungauged HUC-10s to only those with meaningful presence inside the state.

In [10]:
def add_state_area_fraction(huc10_gdf, state_gdf, huc10_id_col, min_fraction=0.10):
    h10 = huc10_gdf.copy()
    st  = state_gdf.copy()

    orig_crs = h10.crs
    h10_proj, _ = ensure_projected_crs(h10, fallback_epsg=3857)
    st_proj, _  = ensure_projected_crs(st,  fallback_epsg=3857)

    h10_proj["huc10_area"] = h10_proj.geometry.area

    intersection = gpd.overlay(
        h10_proj[[huc10_id_col, "geometry"]],
        st_proj[["geometry"]],
        how="intersection",
        keep_geom_type=False,
    )

    if len(intersection) == 0:
        h10_proj["area_in_state"] = 0.0
    else:
        intersection["int_area"] = intersection.geometry.area
        area_in_state = (
            intersection.groupby(huc10_id_col)["int_area"]
            .sum()
            .rename("area_in_state")
            .reset_index()
        )
        h10_proj = h10_proj.merge(area_in_state, on=huc10_id_col, how="left")
        h10_proj["area_in_state"] = h10_proj["area_in_state"].fillna(0.0)

    h10_proj["frac_in_state"] = np.where(
        h10_proj["huc10_area"] > 0,
        h10_proj["area_in_state"] / h10_proj["huc10_area"],
        0.0,
    )
    h10_proj["meets_area_threshold"] = h10_proj["frac_in_state"] >= min_fraction

    if orig_crs is not None and h10_proj.crs != orig_crs:
        return h10_proj.to_crs(orig_crs)
    return h10_proj


def _get_line_downstream_endpoint(geom):
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == "LineString":
        x, y = list(geom.coords)[-1]
        return Point(x, y)
    if geom.geom_type == "MultiLineString":
        parts = list(geom.geoms)
        if not parts:
            return None
        longest = max(parts, key=lambda g: g.length)
        x, y = list(longest.coords)[-1]
        return Point(x, y)
    return None


def add_outlet_in_state_flag(huc10_gdf, flowpaths_gdf, state_gdf, huc10_id_col, hydroseq_col="hydroseq"):
    h10 = huc10_gdf.copy()
    fp  = flowpaths_gdf.copy()
    st  = state_gdf.copy()

    fp_in_huc = gpd.sjoin(
        fp[[hydroseq_col, "geometry"]].copy(),
        h10[[huc10_id_col, "geometry"]].copy(),
        how="inner",
        predicate="intersects",
    ).drop(columns=[c for c in fp.columns if c.startswith("index_")], errors="ignore")

    fp_in_huc[hydroseq_col] = to_numeric_safe(fp_in_huc[hydroseq_col])
    fp_in_huc = fp_in_huc.dropna(subset=[hydroseq_col])
    fp_in_huc = fp_in_huc.sort_values([huc10_id_col, hydroseq_col], ascending=[True, False])

    most_downstream = fp_in_huc.groupby(huc10_id_col).head(1).copy()
    most_downstream["outlet_pt"] = most_downstream.geometry.apply(_get_line_downstream_endpoint)

    outlet_pts = gpd.GeoDataFrame(
        most_downstream[[huc10_id_col]].copy(),
        geometry=most_downstream["outlet_pt"],
        crs=h10.crs,
    ).dropna(subset=["geometry"])

    outlet_pts["outlet_in_state"] = gpd.sjoin(
        outlet_pts,
        st[["geometry"]].copy(),
        how="left",
        predicate="within",
    )["index_right"].notna().values

    h10 = h10.merge(outlet_pts[[huc10_id_col, "outlet_in_state"]], on=huc10_id_col, how="left")
    h10["outlet_in_state"] = h10["outlet_in_state"].fillna(False)
    return h10

## 11. Identify Final Ungauged HUC-10s

HUC-10 polygons are filtered to final ungauged candidates using a two-part rule:

- The HUC-10 must be ungauged (`covered_huc == No`)
- AND it must have sufficient presence in the state, governed by a 50% area threshold applied regardless of whether the outlet is inside or outside the state

In [11]:
AREA_THRESHOLD_OUTLET_INSIDE  = 0.5
AREA_THRESHOLD_OUTLET_OUTSIDE = 0.5

huc10_state = add_state_area_fraction(
    huc10_covered, state, CONFIG["huc10_id_col"], min_fraction=0.10
)

huc10_state = add_outlet_in_state_flag(
    huc10_state, flowpaths_covered, state,
    huc10_id_col=CONFIG["huc10_id_col"],
    hydroseq_col=CONFIG["hydroseq_col"],
)

huc10_state["is_ungauged"]       = huc10_state["covered_huc"] == "No"
huc10_state["outlet_in_state"]   = huc10_state["outlet_in_state"].fillna(False).astype(bool)

huc10_state["is_final_ungauged"] = (
    huc10_state["is_ungauged"]
    & (
        (huc10_state["outlet_in_state"]  & (huc10_state["frac_in_state"] >= AREA_THRESHOLD_OUTLET_INSIDE))
        | (~huc10_state["outlet_in_state"] & (huc10_state["frac_in_state"] >= AREA_THRESHOLD_OUTLET_OUTSIDE))
    )
)

huc10_state["state_area_threshold_used"] = np.where(
    huc10_state["outlet_in_state"],
    AREA_THRESHOLD_OUTLET_INSIDE,
    AREA_THRESHOLD_OUTLET_OUTSIDE,
)

ungauged_huc10_final = huc10_state[huc10_state["is_final_ungauged"]].copy()

print(f"Final ungauged HUC-10 count: {len(ungauged_huc10_final)}")
print(ungauged_huc10_final["outlet_in_state"].value_counts())

Final ungauged HUC-10 count: 5
outlet_in_state
True    5
Name: count, dtype: int64


## 12. Write Output GeoPackage

All result layers are written to the output GeoPackage. An existing file at the same path is removed before writing to avoid layer conflicts.

In [12]:
def write_layers_to_gpkg(output_path, layers_dict):
    out_path = Path(output_path)
    if out_path.exists():
        out_path.unlink()

    for layer_name, gdf in layers_dict.items():
        gdf.to_file(out_path, layer=layer_name, driver="GPKG")

    print(f"Wrote: {out_path.resolve()}")
    print(f"Layers: {list(layers_dict.keys())}")


write_layers_to_gpkg(
    CONFIG["output_gpkg"],
    {
        "flowpaths_gagegap":    flowpaths_covered,
        "divides_gagegap":      divides_covered,
        "huc12_gagegap":        huc12_covered,
        "huc10_gagegap":        huc10_covered,
        "huc10_state":       huc10_state,
        "ungauged_huc10_final": ungauged_huc10_final,
        "gages_assigned":       gages_assigned,
    },
)

Wrote: /home/hpd6c4/Gauge-Gap-Analysis/Gap_output_final_v2.gpkg
Layers: ['flowpaths_gagegap', 'divides_gagegap', 'huc12_gagegap', 'huc10_gagegap', 'huc10_state', 'ungauged_huc10_final', 'gages_assigned']
